# Working with BatchStore

A `BatchStore` is a batch-oriented container for variable-length Python items backed by a single `SChunk`. Each batch is stored in one compressed chunk, and each chunk may contain one or more internal variable-length blocks.

This makes `BatchStore` a good fit when data arrives naturally in batches and you want efficient batch append/update operations together with occasional item-level access inside each batch.

In [1]:
import blosc2


def show(label, value):
    print(f"{label}: {value}")


urlpath = "batchstore_tutorial.b2b"
copy_path = "batchstore_tutorial_copy.b2b"
blosc2.remove_urlpath(urlpath)
blosc2.remove_urlpath(copy_path)

## Creating and populating a BatchStore

A `BatchStore` is indexed by batch. Batches can be appended one by one with `append()` or in bulk with `extend()`. Here we set a small `max_blocksize` just so the internal block structure is easy to observe in `.info`.

In [2]:
store = blosc2.BatchStore(urlpath=urlpath, mode="w", contiguous=True, max_blocksize=2)
store.append(
    [
        {"name": "alpha", "count": 1},
        {"name": "beta", "count": 2},
        {"name": "gamma", "count": 3},
    ]
)
store.append(
    [
        {"name": "delta", "count": 4},
        {"name": "epsilon", "count": 5},
    ]
)
store.extend(
    [
        [{"name": "zeta", "count": 6}],
        [{"name": "eta", "count": 7}, {"name": "theta", "count": 8}],
    ]
)

show("Batches", [batch[:] for batch in store])
show("Number of batches", len(store))

Batches: [[{'name': 'alpha', 'count': 1}, {'name': 'beta', 'count': 2}, {'name': 'gamma', 'count': 3}], [{'name': 'delta', 'count': 4}, {'name': 'epsilon', 'count': 5}], [{'name': 'zeta', 'count': 6}], [{'name': 'eta', 'count': 7}, {'name': 'theta', 'count': 8}]]
Number of batches: 4


## Batch and item access

Indexing the store returns a batch. Indexing a batch returns an item inside that batch. Flat item-wise traversal is available through `iter_items()`.

In [3]:
show("First batch", store[0][:])
show("Second item in first batch", store[0][1])
show("Slice of second batch", store[1][:1])
show("All items", list(store.iter_items()))

First batch: [{'name': 'alpha', 'count': 1}, {'name': 'beta', 'count': 2}, {'name': 'gamma', 'count': 3}]
Second item in first batch: {'name': 'beta', 'count': 2}
Slice of second batch: [{'name': 'delta', 'count': 4}]
All items: [{'name': 'alpha', 'count': 1}, {'name': 'beta', 'count': 2}, {'name': 'gamma', 'count': 3}, {'name': 'delta', 'count': 4}, {'name': 'epsilon', 'count': 5}, {'name': 'zeta', 'count': 6}, {'name': 'eta', 'count': 7}, {'name': 'theta', 'count': 8}]


## Updating, inserting, and deleting batches

Mutation is batch-oriented too: you overwrite, insert, delete, and pop whole batches.

In [4]:
store[1] = [
    {"name": "delta*", "count": 40},
    {"name": "epsilon*", "count": 50},
]
store.insert(1, [{"name": "between", "count": 99}])
removed = store.pop()
del store[0]

show("Popped batch", removed)
show("After updates", [batch[:] for batch in store])

Popped batch: [{'name': 'eta', 'count': 7}, {'name': 'theta', 'count': 8}]
After updates: [[{'name': 'between', 'count': 99}], [{'name': 'delta*', 'count': 40}, {'name': 'epsilon*', 'count': 50}], [{'name': 'zeta', 'count': 6}]]


## Iteration and summary info

Iterating a `BatchStore` yields batches. The `.info` summary reports both batch-level and internal block-level statistics.

In [5]:
show("Batches via iteration", [batch[:] for batch in store])
print(store.info)

Batches via iteration: [[{'name': 'between', 'count': 99}], [{'name': 'delta*', 'count': 40}, {'name': 'epsilon*', 'count': 50}], [{'name': 'zeta', 'count': 6}]]
type       : BatchStore
serializer : msgpack
nbatches   : 3 (items per batch: mean=1.33, max=2, min=1)
nblocks    : 3 (items per block: mean=1.33, max=2, min=1)
nitems     : 4
nbytes     : 84 (84 B)
cbytes     : 468 (468 B)
cratio     : 0.18
cparams    : CParams(codec=<Codec.ZSTD: 5>, codec_meta=0, clevel=5, use_dict=False, typesize=1,
           : nthreads=8, blocksize=0, splitmode=<SplitMode.AUTO_SPLIT: 3>,
           : filters=[<Filter.NOFILTER: 0>, <Filter.NOFILTER: 0>, <Filter.NOFILTER: 0>,
           : <Filter.NOFILTER: 0>, <Filter.NOFILTER: 0>, <Filter.SHUFFLE: 1>], filters_meta=[0,
           : 0, 0, 0, 0, 0], tuner=<Tuner.STUNE: 0>)
dparams    : DParams(nthreads=8)



## Copying and changing storage settings

Like other Blosc2 containers, `BatchStore.copy()` can write a new persistent store while changing storage or compression settings.

In [6]:
store_copy = store.copy(
    urlpath=copy_path,
    contiguous=False,
    cparams={"codec": blosc2.Codec.LZ4, "clevel": 5},
)

show("Copied batches", [batch[:] for batch in store_copy])
show("Copy serializer", store_copy.serializer)
show("Copy codec", store_copy.cparams.codec)

Copied batches: [[{'name': 'between', 'count': 99}], [{'name': 'delta*', 'count': 40}, {'name': 'epsilon*', 'count': 50}], [{'name': 'zeta', 'count': 6}]]
Copy serializer: msgpack
Copy codec: Codec.LZ4


## Round-tripping through cframes and reopening from disk

Tagged persistent stores automatically reopen as `BatchStore`, and a serialized cframe buffer does too.

In [7]:
cframe = store.to_cframe()
restored = blosc2.from_cframe(cframe)
show("from_cframe type", type(restored).__name__)
show("from_cframe batches", [batch[:] for batch in restored])

reopened = blosc2.open(urlpath, mode="r", mmap_mode="r")
show("Reopened type", type(reopened).__name__)
show("Reopened batches", [batch[:] for batch in reopened])

from_cframe type: BatchStore
from_cframe batches: [[{'name': 'between', 'count': 99}], [{'name': 'delta*', 'count': 40}, {'name': 'epsilon*', 'count': 50}], [{'name': 'zeta', 'count': 6}]]
Reopened type: BatchStore
Reopened batches: [[{'name': 'between', 'count': 99}], [{'name': 'delta*', 'count': 40}, {'name': 'epsilon*', 'count': 50}], [{'name': 'zeta', 'count': 6}]]


## Clearing and reusing a store

Calling `clear()` resets the backing storage so the container remains ready for new batches.

In [8]:
scratch = store.copy()
scratch.clear()
scratch.extend(
    [
        [{"name": "fresh", "count": 1}],
        [{"name": "again", "count": 2}, {"name": "done", "count": 3}],
    ]
)
show("After clear + extend", [batch[:] for batch in scratch])

blosc2.remove_urlpath(urlpath)
blosc2.remove_urlpath(copy_path)

After clear + extend: [[{'name': 'fresh', 'count': 1}], [{'name': 'again', 'count': 2}, {'name': 'done', 'count': 3}]]
